# Notebook 07 — Results and Visualization

## What you will learn
1. How to extract displacement, stress, and strain fields from MAPDL
2. How to probe specific points on the mesh
3. How to create animated GIFs from multi-step analyses
4. How to export results to VTK (for ParaView) and CSV
5. How to extract and plot S-parameters from HFSS
6. Von Mises stress — mathematics and physical meaning

---

## Part 1: Stress Measures

### Cauchy stress tensor

The stress state at a point is fully described by the symmetric Cauchy stress tensor:

$$\boldsymbol{\sigma} = \begin{bmatrix} \sigma_x & \tau_{xy} & \tau_{xz} \\ \tau_{xy} & \sigma_y & \tau_{yz} \\ \tau_{xz} & \tau_{yz} & \sigma_z \end{bmatrix}$$

### Principal stresses

The eigenvalues $\sigma_1 \geq \sigma_2 \geq \sigma_3$ of the stress tensor are the
**principal stresses** — normal stresses on planes with zero shear.

$$\det(\boldsymbol{\sigma} - \sigma_i \mathbf{I}) = 0$$

### Von Mises stress (equivalent stress)

The von Mises stress $\sigma_{eq}$ is a scalar measure of the distortion energy:

$$\sigma_{eq} = \sqrt{\frac{3}{2} \mathbf{s} : \mathbf{s}} = \sqrt{\frac{(\sigma_1-\sigma_2)^2 + (\sigma_2-\sigma_3)^2 + (\sigma_3-\sigma_1)^2}{2}}$$

where $\mathbf{s} = \boldsymbol{\sigma} - \frac{1}{3} \text{tr}(\boldsymbol{\sigma}) \mathbf{I}$ is the deviatoric stress.

**Physical meaning:** The von Mises yield criterion states that yielding occurs when:
$$\sigma_{eq} \geq \sigma_y$$

This is the same as requiring the distortion strain energy to exceed a threshold.

In ANSYS: `SEQV` is the von Mises stress (Pa). `S1`, `S2`, `S3` are principal stresses.

In [ ]:
import sys; sys.path.insert(0, '..')
import numpy as np
import matplotlib.pyplot as plt

# ── Compute von Mises stress from a full stress tensor ─────────────────────
def von_mises(sigma_x, sigma_y, sigma_z, tau_xy, tau_yz, tau_xz):
    """Compute von Mises equivalent stress from 6 independent stress components."""
    return np.sqrt(0.5 * (
        (sigma_x - sigma_y)**2 +
        (sigma_y - sigma_z)**2 +
        (sigma_z - sigma_x)**2 +
        6 * (tau_xy**2 + tau_yz**2 + tau_xz**2)
    ))

# Example: uniaxial tension σ_x = 100 MPa, all others = 0
sig_eq = von_mises(100e6, 0, 0, 0, 0, 0)
print(f'Uniaxial tension (σ_x=100 MPa): σ_eq = {sig_eq/1e6:.1f} MPa')
print(f'(Should equal 100 MPa for pure uniaxial state)')

# Pure shear τ_xy = 50 MPa
sig_eq_shear = von_mises(0, 0, 0, 50e6, 0, 0)
print(f'\nPure shear (τ_xy=50 MPa): σ_eq = {sig_eq_shear/1e6:.1f} MPa')
print(f'(Should equal √3 × 50 = {np.sqrt(3)*50:.1f} MPa)')

In [ ]:
# ── Von Mises yield surface visualization ─────────────────────────────────
# In principal stress space (σ_3 = 0 plane)
theta = np.linspace(0, 2*np.pi, 300)
sigma_y = 250e6  # steel yield stress

# Von Mises ellipse in (σ_1, σ_2) plane at σ_3=0
# σ_1² - σ_1 σ_2 + σ_2² = σ_y²
# Parametric form:
s1 = sigma_y * (np.cos(theta) + np.sin(theta)/np.sqrt(3))
s2 = sigma_y * (np.cos(theta) - np.sin(theta)/np.sqrt(3))

# Tresca (hexagon for comparison)
tresca_pts = []
for t in np.linspace(0, 2*np.pi, 7):
    tresca_pts.append([sigma_y * np.cos(t + np.pi/6) * 2/np.sqrt(3),
                        sigma_y * np.sin(t + np.pi/6) * 2/np.sqrt(3)])
tresca_pts = np.array(tresca_pts)

fig, ax = plt.subplots(figsize=(6, 6))
ax.plot(s1/1e6, s2/1e6, 'b-', linewidth=2, label='Von Mises')
ax.plot(tresca_pts[:,0]/1e6, tresca_pts[:,1]/1e6, 'r--', linewidth=1.5, label='Tresca (Hex)')
ax.axhline(0, color='k', linewidth=0.5)
ax.axvline(0, color='k', linewidth=0.5)
ax.set_aspect('equal')
ax.set_xlabel('σ₁ (MPa)'); ax.set_ylabel('σ₂ (MPa)')
ax.set_title(f'Yield Surface in Principal Stress Space (σ_y={sigma_y/1e6:.0f} MPa)')
ax.legend(); ax.grid(True, alpha=0.3)
ax.fill(s1/1e6, s2/1e6, alpha=0.1, color='blue')
ax.text(0, 0, 'Elastic\nregion', ha='center', fontsize=9, color='navy')
plt.tight_layout(); plt.show()

In [ ]:
# ── Load and analyze a sample results CSV ─────────────────────────────────
import pandas as pd
from pathlib import Path

# Try to find a results CSV
csv_paths = list(Path('../outputs').rglob('nodal_results.csv'))

if csv_paths:
    df = pd.read_csv(csv_paths[0])
    print(f'Loaded: {csv_paths[0]}')
    print(f'Columns: {list(df.columns)}')
    print(df.describe())
    
    # Plot displacement distribution
    if 'displacement_norm' in df.columns:
        fig, ax = plt.subplots(figsize=(7, 4))
        ax.hist(df['displacement_norm'] * 1000, bins=50, edgecolor='k', lw=0.4, color='steelblue')
        ax.set_xlabel('|U| (mm)'); ax.set_ylabel('Node count')
        ax.set_title('Displacement Distribution'); ax.grid(True, alpha=0.3)
        plt.tight_layout(); plt.show()
else:
    print('No results CSVs found. Run a simulation first.')
    print('Creating synthetic data for demonstration...')
    
    # Synthetic data for demo
    n = 200
    disp = np.abs(np.random.normal(0.005, 0.002, n))
    vm   = np.abs(np.random.normal(120e6, 30e6, n))
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
    ax1.hist(disp * 1000, bins=30, color='steelblue', edgecolor='k', lw=0.3)
    ax1.set_xlabel('|U| (mm)'); ax1.set_ylabel('Count'); ax1.set_title('Displacement [synthetic]')
    ax2.hist(vm / 1e6, bins=30, color='salmon', edgecolor='k', lw=0.3)
    ax2.set_xlabel('σ_eq (MPa)'); ax2.set_ylabel('Count'); ax2.set_title('Von Mises [synthetic]')
    for ax in [ax1, ax2]: ax.grid(True, alpha=0.3)
    plt.tight_layout(); plt.show()

---

## Part 2: Point Probing

The `probe_point()` function finds the nearest mesh node to a given (x, y, z)
coordinate and returns the field values at that node.

```python
# Example: probe the stress at the hole edge (θ=90° → x=0.05, y=0.01)
vals = probe_point(mapdl, x=0.05, y=0.01, z=0.0,
                   quantities=['von_mises', 'displacement_norm'])
print(f'von Mises at hole edge: {vals["von_mises"]/1e6:.1f} MPa')
# Expected: ~3 × σ_far = 300 MPa (Kirsch solution)
```

## Part 3: EMI Results

S-parameters from HFSS are extracted with `extract_s_parameters(hfss)`.
See `notebooks/09_custom_material_models.ipynb` for the full HFSS workflow.

## Summary

| Task | Function | Output |
|------|----------|--------|
| Extract fields | `extract_results(mapdl, quantities)` | dict of numpy arrays |
| Save CSV | `write_csv(results, output_dir)` | nodal_results.csv |
| Save PNG plots | `save_plots(mapdl, results, output_dir)` | PNG files |
| Export VTK | `export_vtk(mapdl, output_dir)` | results.vtk for ParaView |
| Probe a point | `probe_point(mapdl, x, y, z)` | dict of values at node |
| Animate | `save_animation(mapdl, output_dir)` | animation.gif |

**Next:** `08_multiphysics_pipeline.ipynb`